# Evaluation of the Usage Extraction Module
This notebook contains the code for evaluation in section 5.1 Usage Extraction Module of the *FairGround* paper.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import ast
import re

In [ ]:
working_dir = Path("../experiments/fair-eval")
manual_annotations_path = "../manual_annotations/fairness/test_manual_dataset_table_clean.csv"
# load the intermediate dataset usages table
dataset_usages_table_path = working_dir / "fg/output/dataset_usages_table.csv"

In [3]:
# we will save evaluation details in the evaluation directory
evaluation_dir = working_dir / "evaluation"
evaluation_dir.mkdir(parents=True, exist_ok=True)

In [4]:
# load manual (ground truth) datasets and automatically identified datasets
manual_df = pd.read_csv(manual_annotations_path)
automatic_df = pd.read_csv(dataset_usages_table_path)

### Helper functions for evaluation

In [5]:
from difflib import get_close_matches
from difflib import SequenceMatcher
from dataset_extraction.state.paper_entries import canonicalize_title

def _normalize_name(name):
    return name.lower().replace(" ", "_").replace("-", "_")
def _get_normalized_aliases_list(name, other_aliases):
    aliases = []
    if pd.notna(name):
        aliases.append(_normalize_name(name))
    if pd.notna(other_aliases):
        if other_aliases.startswith("["):
            other_aliases = ast.literal_eval(other_aliases)
        else:
            other_aliases = other_aliases.split("; ")
        for a in other_aliases:
            aliases.append(_normalize_name(a))
    return aliases
def _is_substr(s, t):
    return s in t

## Match datasets
The following code creates a mapping between each manually identified dataset and a list of matched automatically identified datasets rows in the dataset usage table.
We use  Python's difflib fuzzy matching algorithm. To be considered a match, a manual and automatic dataset must:
1. have a similarity score greater than 0.6 for the dataset's names / aliases OR have the automatic/manual dataset names be a substring of another, AND
2. have a similarity score greath than 0.6 for the paper citation titles.

In [6]:
dataset_matches = {}

for m_idx, m_row in manual_df.iterrows():
    m_aliases = _get_normalized_aliases_list(m_row["dataset_name"], m_row["dataset_aliases"])
    m_title = m_row["citation_title"]
    dataset_matches[m_idx] = []
    for a_idx, a_row in automatic_df.iterrows():
        a_aliases = _get_normalized_aliases_list(a_row["dataset_full_identifier"], a_row["aliases"])
        aliases_sim = np.array([
            [
                (
                    SequenceMatcher(None, m_a, a_a).ratio() > 0.6 or
                    _is_substr(m_a, a_a) or _is_substr(a_a, m_a)
                )
                for a_a in a_aliases
            ]
            for m_a in m_aliases
        ])
        if np.any(aliases_sim):
            a_title = a_row["citation_title"]
            if (pd.isna(m_title) or pd.isna(a_title) or 
                SequenceMatcher(
                    None, canonicalize_title(m_title), 
                    canonicalize_title(a_title)
                ).ratio() > 0.6
            ):
                dataset_matches[m_idx].append(a_idx)

We convert the match dictionary into json format and save it to the evaluation directory.

In [7]:
def clean_nan(value):
    if isinstance(value, list):
        return [None if pd.isna(x) else x for x in value]
    return None if pd.isna(value) else value

eval_matches = {}
for m_idx, a_indicies in dataset_matches.items():
    m_name = clean_nan(manual_df["dataset_name"].iloc[m_idx])
    m_title = clean_nan(manual_df["citation_title"].iloc[m_idx])
    a_names = automatic_df["dataset_full_identifier"].iloc[list(a_indicies)].tolist()
    a_names = clean_nan(a_names)
    a_titles = automatic_df["citation_title"].iloc[list(a_indicies)].tolist()
    a_titles = clean_nan(a_titles)
    eval_matches[str((m_idx, m_name, m_title))] = [str(item) for item in list(zip(a_indicies, a_names, a_titles))]
with open(evaluation_dir / "test.txt", "w") as f:
    json.dump(eval_matches, f, indent=2)
    

Now that we have a matched each dataset by name and citation title, we conduct further manual checking of whether the datasets are matched semantically. We save the matches that pass our manual checks to `{evaluation_dir}/test_manual.txt`. 

In [8]:
with open(evaluation_dir / "test_manual.txt", "r") as f:
    eval_matches = json.load(f)

Finally, we calculate the proportion of manually identified datasets with automatic matches.

In [9]:
match_cnt = 0
for k, v in eval_matches.items():
    m_idx, m_name, m_title = ast.literal_eval(k)
    m_full_id = manual_df["full_dataset_id"].iloc[m_idx]
    matches =[ast.literal_eval(vv) for vv in v]
    if len(matches)>0:
        match_cnt = match_cnt + 1
print(f"Total matches: {match_cnt}/{len(eval_matches)} = {match_cnt/len(eval_matches)}")

Total matches: 108/130 = 0.8307692307692308
